# Fine-tune MediaPipe BlazeFace for general face detection (70/20/10 split)

Knowing Eye's live monitoring already detects faces with **MediaPipe BlazeFace (short-range)**: the browser
overlay loads `frontend/public/models/blaze_face_short_range.tflite`, and the server's
`backend/ai/models/face_landmarker.task` contains the very same detector (byte-identical file). This notebook
fine-tunes that model on the project's face dataset and produces the paper's face-detection tables and figures
for it.

It uses **the same data and split as `train_eval_face_70_20_10.ipynb`** (YOLOv8n / YOLO11n / RT-DETR-l):
`face_split_70_20_10.zip`, built by `scripts/split_face_dataset.py` (WIDER FACE event categories + FDDB, seed 42,
exact duplicates removed). Same images, same split, same metric code, so BlazeFace can sit in the same table as
the YOLO models.

| Split | Images | Used for |
|---|---|---|
| train (70%) | 2,809 | fitting the weights |
| val (20%) | 803 | early stopping + picking the best epoch (seen indirectly - **not** reported as final accuracy) |
| test (10%) | 402 | evaluated **once**, after training - **these are the numbers for the paper** |

**How BlazeFace is fine-tuned.** MediaPipe Model Maker has no face-detector task, so this notebook:
1. rebuilds Google's `blaze_face_short_range.tflite` layer by layer as a Keras model carrying the pretrained
   weights (checked against the original: same outputs to ~1e-5);
2. trains it with SSD-style anchor matching that reproduces how the pretrained model already assigns faces to its
   896 anchors. WIDER FACE / FDDB have no facial landmarks, so the 6 BlazeFace keypoints are kept where the
   original model puts them (distillation from a frozen copy);
3. evaluates with MediaPipe's exact pre/post-processing (letterbox, anchor decoding, weighted NMS) and
   Ultralytics' exact metric code (same Precision / Recall / F1 / mAP definitions as the YOLO tables);
4. exports a drop-in `.tflite` (same input, outputs and metadata as Google's) plus a `face_landmarker.task` with
   the fine-tuned detector inside, and verifies both by loading them in MediaPipe.

**Before running**
1. Upload `face_split_70_20_10.zip` to your Google Drive root (`MyDrive/face_split_70_20_10.zip`) - the same
   file the YOLO notebook uses, so skip this if it is already there.
2. Runtime -> Change runtime type -> **T4 GPU** (a CPU runtime also works; BlazeFace is tiny, it is just slower).
3. Runtime -> Run all. Expect very roughly 30-60 minutes on a T4 (BlazeFace trains fast; most of each epoch is
   scoring the validation set). Training saves to Drive after every epoch; if the session dies, Run all again and
   it resumes where it stopped.

**Rule:** do not change any setting after looking at test results and then re-test. The test set is only honest
if it is used once.

In [ ]:
import os, sys, subprocess

# oneDNN's CPU graph fusions crash on this model inside tf.function (seen on TF 2.21). GPU runs are unaffected;
# this only matters on a CPU runtime, and must be set before TensorFlow is imported.
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # TensorFlow/Keras are preinstalled on Colab; MediaPipe is not. Its Python API needs the EGL/GLES libraries.
    subprocess.run('apt-get -qq install -y libegl1 libgles2 > /dev/null', shell=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe==1.1.1'], check=True)

import hashlib, json, math, random, re, shutil, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import keras
import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from mediapipe.tasks.python import BaseOptions, vision
from tensorflow.lite.python import schema_py_generated as schema
from tensorflow.lite.tools import flatbuffer_utils

try:   # tf.lite.Interpreter is being moved to the ai_edge_litert package
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter
trapezoid = getattr(np, 'trapezoid', None) or np.trapz   # NumPy 2 renamed trapz

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)   # leave GPU memory for PyTorch (YOLO comparison)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| MediaPipe', mp.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'none - running on CPU')

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/face_split_70_20_10.zip'
DATA_ROOT = '/content/data'
OUT_DIR = '/content/drive/MyDrive/knowing-eye-blazeface-70-20-10'   # everything that must survive the session
YOLO_DIR = '/content/drive/MyDrive/knowing-eye-70-20-10'            # outputs of train_eval_face_70_20_10.ipynb
EXPECTED_COUNTS = {'train': 2809, 'val': 803, 'test': 402}

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = f'{DATA_ROOT}/face_split_70_20_10'
if not Path(DATA_DIR, 'images').exists():
    zipfile.ZipFile(DATASET_ZIP).extractall(DATA_ROOT)   # local disk: much faster than reading from Drive
counts = {s: len(os.listdir(f'{DATA_DIR}/images/{s}')) for s in ('train', 'val', 'test')}
print(counts)
assert counts == EXPECTED_COUNTS, 'Unexpected split sizes - wrong zip?'

In [ ]:
# ---- Config ----
EPOCHS = 100
BATCH = 32
LR = 1e-3          # Adam peak learning rate: 3-epoch warm-up, then cosine decay to 1% of it
PATIENCE = 20      # early stopping on the VAL split (same as the YOLO runs)
SEED = 42
TRAIN_EVAL_N = 800 # fixed, un-augmented training images scored after every epoch ("training accuracy")

# BlazeFace short-range looks at a 128x128 picture of the whole frame. Google documents it for faces within
# ~2 m of the camera; at 128 px a face under ~8% of the image side is only a few pixels and cannot be found.
# Results are therefore reported twice: on ALL faces (same protocol as the YOLO table) and on WEBCAM-RANGE
# faces (at least IN_RANGE_MIN_FACE of the image's longer side; smaller faces count as neither hit nor miss).
IN_RANGE_MIN_FACE = 0.10
DEPLOY_CONF = 0.5     # MediaPipe's default min_detection_confidence (what the frontend uses)
REALTIME_FPS = 30     # threshold line on the accuracy-vs-speed chart

FIG_DIR = f'{OUT_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)
keras.utils.set_random_seed(SEED)

# Google's pretrained models - byte-identical to the copies in the repository.
MODELS = {
    'blaze_face_short_range.tflite': (
        'https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/latest/'
        'blaze_face_short_range.tflite',
        'b4578f35940bf5a1a655214a1cce5cab13eba73c1297cd78e1a04c2380b0152f'),
    'face_landmarker.task': (
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/'
        'face_landmarker.task',
        '64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff'),
}
for name, (url, sha) in MODELS.items():
    path = Path(OUT_DIR, 'pretrained', name)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f'{name}: {"OK, same as the repo copy" if digest == sha else "WARNING - differs from the repo copy"}')
BASE_TFLITE = f'{OUT_DIR}/pretrained/blaze_face_short_range.tflite'
BASE_TASK = f'{OUT_DIR}/pretrained/face_landmarker.task'

## 1. BlazeFace as a trainable Keras model

The `.tflite` file is a frozen graph of 164 operations (convolutions, depthwise convolutions, residual adds,
ReLUs, channel padding, max-pooling). `blazeface_from_tflite` walks that graph and recreates each operation as
the equivalent Keras layer, copying Google's weights in. The result is the original network - 16 BlazeBlocks,
101,390 parameters, input 128x128 RGB in [-1, 1], outputs `regressors` (896 x 16: box + 6 keypoints per
anchor) and `classificators` (896 x 1: face logit per anchor) - except now it can be trained.

`export_tflite` goes the other way: Keras -> float16 `.tflite` with the original's input/output names and order,
and the original's MediaPipe metadata (input normalisation etc.) copied in, so MediaPipe loads it exactly like
Google's file.

In [ ]:
INPUT = 128
OP_NAMES = {v: k for k, v in vars(schema.BuiltinOperator).items() if not k.startswith('_')}
NP_TYPES = {schema.TensorType.FLOAT32: np.float32, schema.TensorType.FLOAT16: np.float16,
            schema.TensorType.INT32: np.int32}


@keras.saving.register_keras_serializable('blazeface')
class ChannelPad(keras.layers.Layer):
    """Zero-pads the channel axis so a residual branch matches a wider block (TFLite PAD op)."""

    def __init__(self, pad, **kwargs):
        super().__init__(**kwargs)
        self.pad = pad

    def call(self, x):
        return keras.ops.pad(x, [[0, 0], [0, 0], [0, 0], [0, self.pad]])

    def get_config(self):
        return {**super().get_config(), 'pad': self.pad}


def blazeface_from_tflite(path, batch_size=None):
    """Rebuild the .tflite graph op by op as a Keras model carrying the pretrained weights."""
    fb = flatbuffer_utils.read_model(path)
    g = fb.subgraphs[0]
    source = {}  # DEQUANTIZE output tensor -> its float16 constant

    def const(i):
        t = g.tensors[source.get(i, i)]
        return np.frombuffer(bytes(fb.buffers[t.buffer].data), NP_TYPES[t.type]).reshape(t.shape)

    def name_of(i):
        return re.sub(r'\W', '_', g.tensors[i].name.decode()).replace('__xeno_compat__1', '')

    def padding(opt):
        return 'same' if opt.padding == schema.Padding.SAME else 'valid'

    inp = keras.Input(tuple(g.tensors[g.inputs[0]].shape[1:]), batch_size=batch_size, name='input')
    x = {g.inputs[0]: inp}
    weights = []
    for op in g.operators:
        code = fb.operatorCodes[op.opcodeIndex]
        kind = OP_NAMES[max(code.builtinCode, code.deprecatedBuiltinCode)]
        ins, out, opt = list(op.inputs), op.outputs[0], op.builtinOptions
        name = name_of(out)
        if kind == 'DEQUANTIZE':
            source[out] = ins[0]
            continue
        if kind in ('CONV_2D', 'DEPTHWISE_CONV_2D', 'ADD'):
            assert opt.fusedActivationFunction == schema.ActivationFunctionType.NONE, name
        if kind == 'CONV_2D':
            k, b = const(ins[1]).astype(np.float32), const(ins[2]).astype(np.float32)
            layer = keras.layers.Conv2D(k.shape[0], k.shape[1:3], strides=(opt.strideH, opt.strideW),
                                        padding=padding(opt), name=name)
            weights.append((layer, [k.transpose(1, 2, 3, 0), b]))
            ins = ins[:1]
        elif kind == 'DEPTHWISE_CONV_2D':
            k, b = const(ins[1]).astype(np.float32), const(ins[2]).astype(np.float32)
            layer = keras.layers.DepthwiseConv2D(k.shape[1:3], strides=(opt.strideH, opt.strideW),
                                                 padding=padding(opt), name=name)
            weights.append((layer, [k.reshape(k.shape[1], k.shape[2], -1, 1), b]))
            ins = ins[:1]
        elif kind == 'RELU':
            layer = keras.layers.ReLU(name=name)
        elif kind == 'ADD':
            layer = keras.layers.Add(name=name)
        elif kind == 'PAD':
            pads = const(ins[1])
            assert not pads[:3].any() and pads[3, 0] == 0, f'{name}: only channel padding is supported'
            layer, ins = ChannelPad(int(pads[3, 1]), name=name), ins[:1]
        elif kind == 'MAX_POOL_2D':
            layer = keras.layers.MaxPooling2D((opt.filterHeight, opt.filterWidth), (opt.strideH, opt.strideW),
                                              padding=padding(opt), name=name)
        elif kind == 'RESHAPE':
            layer, ins = keras.layers.Reshape(tuple(g.tensors[out].shape[1:]), name=name), ins[:1]
        elif kind == 'CONCATENATION':
            layer = keras.layers.Concatenate(axis=opt.axis, name=name)
        else:
            raise NotImplementedError(f'TFLite op {kind} ({name})')
        args = [x[i] for i in ins]
        x[out] = layer(args if len(args) > 1 else args[0])
    model = keras.Model(inp, [x[i] for i in g.outputs], name='blazeface_short_range')
    for layer, w in weights:
        layer.set_weights(w)
    return model


def export_tflite(model, out_path, reference=None):
    """Keras -> float16 .tflite with the reference model's I/O names, output order and MediaPipe metadata."""
    reference = reference or BASE_TFLITE
    fixed = blazeface_from_tflite(reference, batch_size=1)   # static shapes -> plain RESHAPE ops
    fixed.set_weights(model.get_weights())
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(fixed)
    except Exception:   # some TF/Keras 3 combinations: go through a SavedModel instead (same result)
        saved = f'/tmp/blazeface_export_{os.getpid()}'
        fixed.export(saved, format='tf_saved_model', verbose=False)
        converter = tf.lite.TFLiteConverter.from_saved_model(saved)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    fb = flatbuffer_utils.read_model_from_bytearray(bytearray(converter.convert()))

    ref = flatbuffer_utils.read_model(reference)
    g, rg = fb.subgraphs[0], ref.subgraphs[0]
    # MediaPipe reads outputs by position: [regressors (1,896,16), classificators (1,896,1)].
    if [list(g.tensors[i].shape) for i in g.outputs] == [[1, 896, 1], [1, 896, 16]]:
        g.outputs = g.outputs[::-1]
    assert [list(g.tensors[i].shape) for i in g.outputs] == [[1, 896, 16], [1, 896, 1]]
    for mine, theirs in zip(list(g.inputs) + list(g.outputs), list(rg.inputs) + list(rg.outputs)):
        g.tensors[mine].name = rg.tensors[theirs].name
    for sig in fb.signatureDefs or []:
        sig.outputs.sort(key=lambda t: list(g.outputs).index(t.tensorIndex))
    # MediaPipe refuses a float model without input-normalisation metadata: copy Google's.
    meta = next(m for m in ref.metadata if m.name == b'TFLITE_METADATA')
    buf = schema.BufferT()
    buf.data = ref.buffers[meta.buffer].data
    fb.buffers.append(buf)
    entry = schema.MetadataT()
    entry.name, entry.buffer = b'TFLITE_METADATA', len(fb.buffers) - 1
    fb.metadata = [m for m in (fb.metadata or []) if m.name != b'TFLITE_METADATA'] + [entry]
    flatbuffer_utils.write_model(fb, out_path)
    return out_path


def repack_face_landmarker(task_in, detector_tflite, task_out):
    """Copy of face_landmarker.task with its face_detector.tflite replaced."""
    with zipfile.ZipFile(task_in) as src, zipfile.ZipFile(task_out, 'w', zipfile.ZIP_STORED) as dst:
        for item in src.infolist():
            data = Path(detector_tflite).read_bytes() if item.filename == 'face_detector.tflite' else src.read(item)
            dst.writestr(item, data)
    return task_out


teacher = blazeface_from_tflite(BASE_TFLITE)   # frozen original: baseline + landmark targets
teacher.trainable = False
print(f'BlazeFace short-range rebuilt: {len(teacher.layers)} layers, {teacher.count_params():,} parameters')

## 2. MediaPipe-identical pre- and post-processing

To make the reported numbers describe the model *as MediaPipe runs it*, evaluation copies MediaPipe's
`FaceDetector` pipeline exactly:

* **Letterbox** - the whole frame is fitted into 128x128 keeping its aspect ratio, centred, black border, with one
  bilinear `warpAffine` (no anti-aliasing). A plain `cv2.resize` gives noticeably different scores.
* **Anchors** - 896 SSD anchors: a 16x16 grid with 2 anchors per cell and an 8x8 grid with 6 per cell, in
  MediaPipe's order. MediaPipe decodes with fixed-size anchors, so box size comes straight from the regressor.
* **Decoding** - `box centre = anchor centre + offset / 128`, `box size = output / 128`, score = sigmoid of the
  logit clipped to +-100.
* **Weighted NMS** at IoU 0.3 - overlapping boxes are *averaged* (weighted by score), not just discarded.

The sanity-check cell below confirms this matches MediaPipe's own output on real images.

In [ ]:
# MediaPipe face_detector_graph, short-range model: SSD anchor options.
SSD = dict(num_layers=4, min_scale=0.1484375, max_scale=0.75, strides=[8, 16, 16, 16])


def ssd_anchors():
    """(896, 3) [x_centre, y_centre, nominal_scale] in MediaPipe's anchor order.

    MediaPipe decodes with fixed_anchor_size=True (anchor w = h = 1); the nominal scale is only used for
    deciding which anchors are responsible for which face size during training.
    """
    n, lo, hi, strides = SSD['num_layers'], SSD['min_scale'], SSD['max_scale'], SSD['strides']
    scale = lambda i: lo + (hi - lo) * i / (n - 1)
    anchors, layer = [], 0
    while layer < n:
        scales, last = [], layer
        while last < n and strides[last] == strides[layer]:
            nxt = 1.0 if last == n - 1 else scale(last + 1)
            scales += [scale(last), math.sqrt(scale(last) * nxt)]
            last += 1
        fm = math.ceil(INPUT / strides[layer])
        anchors += [((x + 0.5) / fm, (y + 0.5) / fm, s) for y in range(fm) for x in range(fm) for s in scales]
        layer = last
    return np.array(anchors, np.float32)


ANCHORS = ssd_anchors()
assert len(ANCHORS) == 896


def letterbox(img_rgb, size=INPUT):
    """MediaPipe ImageToTensor(keep_aspect_ratio, BORDER_ZERO): centred square ROI, one bilinear warpAffine.

    Returns the uint8 input and (scale_x, scale_y, pad_x, pad_y) with tensor_xy = pad + image_xy * scale.
    """
    h, w = img_rgb.shape[:2]
    s = max(h, w) / size
    m = np.array([[1 / s, 0, (max(h, w) - w) / 2 / s], [0, 1 / s, (max(h, w) - h) / 2 / s]], np.float32)
    out = cv2.warpAffine(img_rgb, m, (size, size), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return out, (w / s / size, h / s / size, m[0, 2] / size, m[1, 2] / size)


def normalize(batch_uint8):
    return batch_uint8.astype(np.float32) / 127.5 - 1.0


def decode(regressors, classificators):
    """Raw (896,16), (896,1) -> boxes xyxy (input-normalised), keypoints (896,6,2), scores."""
    r = regressors / INPUT
    cx, cy = ANCHORS[:, 0] + r[:, 0], ANCHORS[:, 1] + r[:, 1]
    boxes = np.stack([cx - r[:, 2] / 2, cy - r[:, 3] / 2, cx + r[:, 2] / 2, cy + r[:, 3] / 2], 1)
    kps = r[:, 4:].reshape(-1, 6, 2) + ANCHORS[:, None, :2]
    scores = 1 / (1 + np.exp(-np.clip(classificators[:, 0].astype(np.float64), -100, 100)))
    return boxes, kps, scores


def iou_matrix(a, b):
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    area = lambda x: np.clip(x[:, 2] - x[:, 0], 0, None) * np.clip(x[:, 3] - x[:, 1], 0, None)
    union = area(a)[:, None] + area(b)[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)


def weighted_nms(boxes, kps, scores, min_score, iou_thresh=0.3):
    """MediaPipe NonMaxSuppressionCalculator, algorithm WEIGHTED, overlap IoU."""
    order = np.argsort(-scores)
    order = order[scores[order] >= min_score]
    out_b, out_k, out_s = [], [], []
    while order.size:
        sim = iou_matrix(boxes[order[:1]], boxes[order])[0]
        group = order[sim > iou_thresh]
        group = group if group.size else order[:1]
        w = scores[group][:, None]
        out_b.append((boxes[group] * w).sum(0) / w.sum())
        out_k.append((kps[group] * w[:, :, None]).sum(0) / w.sum())
        out_s.append(scores[order[0]])
        if not (sim > iou_thresh).any():
            break   # MediaPipe stops when an iteration removes nothing
        order = order[sim <= iou_thresh]
    if not out_s:
        return np.zeros((0, 4), np.float32), np.zeros((0, 6, 2), np.float32), np.zeros(0, np.float32)
    return np.array(out_b, np.float32), np.array(out_k, np.float32), np.array(out_s, np.float32)


def unletterbox(boxes, lb):
    sx, sy, px, py = lb
    return (boxes - [px, py, px, py]) / [sx, sy, sx, sy]


def to_tensor_space(boxes, lb):
    sx, sy, px, py = lb
    return boxes * [sx, sy, sx, sy] + [px, py, px, py]


def predict(model, x_uint8, batch=256):
    regs, clss = [], []
    for i in range(0, len(x_uint8), batch):
        r, c = model(normalize(x_uint8[i:i + batch]), training=False)
        regs.append(r.numpy()); clss.append(c.numpy())
    return np.concatenate(regs), np.concatenate(clss)


def detect(reg, cls, lbs, min_score=0.01, iou_thresh=0.3, top_k=300):
    """Per image: decode + weighted NMS, back to image-normalised xyxy, clipped to the image."""
    out = []
    for r, c, lb in zip(reg, cls, lbs):
        boxes, kps, scores = decode(r, c)
        # MediaPipe drops boxes with non-positive width/height before NMS.
        scores = np.where((boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1]), scores, -1.0)
        keep = np.argsort(-scores)[:top_k]
        keep = keep[scores[keep] >= min_score]
        b, _, s = weighted_nms(boxes[keep], kps[keep], scores[keep], min_score, iou_thresh)
        out.append((np.clip(unletterbox(b, lb), 0, 1), s))
    return out

In [ ]:
# Sanity checks (validation images only - the test split stays untouched until the end).
_val_paths = sorted(str(p) for p in Path(DATA_DIR, 'images', 'val').iterdir())[:25]

# (a) Keras rebuild == original .tflite
interp = Interpreter(BASE_TFLITE)
interp.allocate_tensors()
worst = 0.0
for p in _val_paths[:5]:
    x = normalize(letterbox(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))[0][None])
    interp.set_tensor(interp.get_input_details()[0]['index'], x)
    interp.invoke()
    ref = [interp.get_tensor(d['index']) for d in interp.get_output_details()]
    ours = [t.numpy() for t in teacher(x, training=False)]
    worst = max(worst, *(np.abs(a - b).max() / np.abs(a).max() for a, b in zip(ref, ours)))
print(f'(a) Keras rebuild vs original .tflite: max relative difference {worst:.1e}')
assert worst < 1e-4


# (b) our letterbox + decode + weighted NMS == MediaPipe FaceDetector
def mediapipe_detect(model_path, paths, min_conf=DEPLOY_CONF):
    det = vision.FaceDetector.create_from_options(vision.FaceDetectorOptions(
        base_options=BaseOptions(model_asset_path=model_path), min_detection_confidence=min_conf))
    out = []
    for p in paths:
        rgb = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]
        res = det.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        out.append(np.array([[d.bounding_box.origin_x / w, d.bounding_box.origin_y / h,
                              (d.bounding_box.origin_x + d.bounding_box.width) / w,
                              (d.bounding_box.origin_y + d.bounding_box.height) / h,
                              d.categories[0].score] for d in res.detections]).reshape(-1, 5))
    det.close()
    return out


def compare_with_mediapipe(model, model_path, paths):
    """Returns (#MediaPipe detections, #ours, mean |score difference|, mean IoU) over matched faces."""
    mp_dets = mediapipe_detect(model_path, paths)
    x, lbs = zip(*(letterbox(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)) for p in paths))
    ours = detect(*predict(model, np.stack(x)), lbs, min_score=DEPLOY_CONF)
    n_mp = n_ours = 0
    dscore, ious = [], []
    for m, (b, s) in zip(mp_dets, ours):
        n_mp, n_ours = n_mp + len(m), n_ours + len(b)
        if len(m) and len(b):
            iou = iou_matrix(np.clip(m[:, :4], 0, 1), b)
            for i, j in enumerate(iou.argmax(1)):
                dscore.append(abs(m[i, 4] - s[j])); ious.append(iou[i, j])
    return n_mp, n_ours, float(np.mean(dscore)) if dscore else 0.0, float(np.mean(ious)) if ious else 1.0


n_mp, n_ours, dscore, miou = compare_with_mediapipe(teacher, BASE_TFLITE, _val_paths)
print(f'(b) MediaPipe FaceDetector vs this notebook on {len(_val_paths)} images: {n_mp} vs {n_ours} faces, '
      f'mean |score diff| {dscore:.4f}, mean IoU {miou:.3f}')

## 3. Data

* **Training images** are decoded once and kept in RAM (longer side 448 px). Every epoch each image gets a fresh
  random **zoom/shift crop** (0.8x-2.5x - mostly closer, webcam-like framing), **horizontal flip** and
  **colour jitter** (brightness, contrast, saturation, occasional greyscale), rendered with the same
  `warpAffine` MediaPipe uses so training and deployment see the same kind of 128x128 picture.
* **Validation / test / train-eval images** are letterboxed once from the full-resolution file, exactly as
  MediaPipe would, and never augmented.

In [ ]:
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
CACHE_SIDE = 448
ZOOM = (0.8, 2.5)


def read_yolo_labels(path):
    """YOLO 'cls cx cy w h' (normalised) -> (n, 4) xyxy normalised to the image."""
    rows = [l.split() for l in Path(path).read_text().splitlines() if l.strip()] if Path(path).exists() else []
    a = np.array([[float(v) for v in r[1:5]] for r in rows], np.float32).reshape(-1, 4)
    return np.stack([a[:, 0] - a[:, 2] / 2, a[:, 1] - a[:, 3] / 2, a[:, 0] + a[:, 2] / 2, a[:, 1] + a[:, 3] / 2], 1)


def list_split(split):
    images = sorted(p for p in Path(DATA_DIR, 'images', split).iterdir() if p.suffix.lower() in IMAGE_EXT)
    return [(str(p), read_yolo_labels(Path(DATA_DIR, 'labels', split, p.stem + '.txt'))) for p in images]


def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f'cannot read {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def load_train_cache(items):
    def load(item):
        img = read_rgb(item[0])
        r = CACHE_SIDE / max(img.shape[:2])
        if r < 1:
            img = cv2.resize(img, (round(img.shape[1] * r), round(img.shape[0] * r)), interpolation=cv2.INTER_AREA)
        return img, item[1]
    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(load, items))


def load_eval_set(items):
    def load(item):
        lb, info = letterbox(read_rgb(item[0]))
        return lb, info
    with ThreadPoolExecutor(8) as ex:
        out = list(ex.map(load, items))
    return dict(x=np.stack([o[0] for o in out]), lb=[o[1] for o in out], gt=[i[1] for i in items],
                paths=[i[0] for i in items])


def augment(img, boxes, rng):
    """Random zoom/shift crop + flip (one warpAffine, like MediaPipe) + colour jitter.

    Returns the 128x128 uint8 input, the faces to learn and the faces to ignore (cut off by the crop),
    both as xyxy in 0..1 input coordinates.
    """
    h, w = img.shape[:2]
    roi = max(h, w) / math.exp(rng.uniform(*np.log(ZOOM)))
    lo_x, hi_x = sorted((roi / 2, w - roi / 2))
    lo_y, hi_y = sorted((roi / 2, h - roi / 2))
    cx, cy = rng.uniform(lo_x, hi_x), rng.uniform(lo_y, hi_y)
    s = roi / INPUT
    m = np.array([[1 / s, 0, -(cx - roi / 2) / s], [0, 1 / s, -(cy - roi / 2) / s]], np.float32)
    if rng.random() < 0.5:
        m[0] = [-m[0, 0], 0, INPUT - m[0, 2]]   # horizontal flip
    out = cv2.warpAffine(img, m, (INPUT, INPUT), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=0)

    b = boxes * [w, h, w, h]
    xs = (b[:, [0, 2]] * m[0, 0] + m[0, 2]) / INPUT
    ys = (b[:, [1, 3]] * m[1, 1] + m[1, 2]) / INPUT
    full = np.stack([xs.min(1), ys.min(1), xs.max(1), ys.max(1)], 1)
    clipped = np.clip(full, 0, 1)
    area = lambda a: np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    visible = area(clipped) / np.maximum(area(full), 1e-9)
    faces, cut = clipped[visible >= 0.6], clipped[(visible > 0) & (visible < 0.6)]

    f = out.astype(np.float32)
    f = (f - f.mean()) * rng.uniform(0.7, 1.3) + f.mean() * rng.uniform(0.7, 1.3)   # contrast, brightness
    grey = f.mean(axis=2, keepdims=True)
    f = grey + (f - grey) * (0.0 if rng.random() < 0.05 else rng.uniform(0.6, 1.4))  # saturation / greyscale
    return np.clip(f, 0, 255).astype(np.uint8), faces, cut


t0 = time.time()
train_items = list_split('train')
train_cache = load_train_cache(train_items)
val_set = load_eval_set(list_split('val'))
te_idx = sorted(np.random.default_rng(SEED).permutation(len(train_items))[:TRAIN_EVAL_N])
train_eval_set = load_eval_set([train_items[i] for i in te_idx])
print(f'loaded in {time.time() - t0:.0f}s: {len(train_cache)} train, {len(val_set["x"])} val, '
      f'{len(train_eval_set["x"])} train-eval images')

## 4. Training targets: which anchors should fire for which face

Each of the 896 anchors has a nominal square size from MediaPipe's SSD settings (0.15 and 0.23 of the frame on
the 16x16 grid; 0.35 to 0.87 on the 8x8 grid). Measured on WIDER FACE images, the pretrained model fires on
the anchors whose nominal square overlaps the face by roughly IoU 0.4 or more, so training uses that rule:

* **face anchor**: IoU >= 0.4 with a face (and every face gets at least its single best anchor);
* **background anchor**: IoU < 0.3 with every face;
* **ignored**: in between, and around faces too small to learn (< 6% of the frame - a few pixels at 128x128) or
  mostly cut off by the crop.

Because this reproduces the pretrained model's own convention, fine-tuning refines Google's detector instead of
re-learning it from scratch.

In [ ]:
POS_IOU, NEG_IOU = 0.4, 0.3
MIN_TRAIN_FACE = 0.06
NOMINAL = np.stack([ANCHORS[:, 0] - ANCHORS[:, 2] / 2, ANCHORS[:, 1] - ANCHORS[:, 2] / 2,
                    ANCHORS[:, 0] + ANCHORS[:, 2] / 2, ANCHORS[:, 1] + ANCHORS[:, 2] / 2], 1)
HALF_STRIDE = np.where(np.arange(896) < 512, 0.5 / 16, 0.5 / 8).astype(np.float32)


def encode(faces, ignore):
    """Per-anchor label (1 face, 0 background, -1 ignored) and its target box (xyxy, input-normalised)."""
    faces = faces.astype(np.float32).reshape(-1, 4)
    ignore = ignore.astype(np.float32).reshape(-1, 4)
    size = np.sqrt(np.clip(faces[:, 2] - faces[:, 0], 0, None) * np.clip(faces[:, 3] - faces[:, 1], 0, None))
    ignore = np.concatenate([ignore, faces[size < MIN_TRAIN_FACE]])
    faces = faces[size >= MIN_TRAIN_FACE]
    labels = np.zeros(896, np.float32)
    targets = np.zeros((896, 4), np.float32)
    if len(faces):
        iou = iou_matrix(NOMINAL, faces)
        best_face, best_iou = iou.argmax(1), iou.max(1)
        labels[best_iou >= NEG_IOU] = -1
        labels[best_iou >= POS_IOU] = 1
        forced = iou.argmax(0)
        labels[forced] = 1
        best_face[forced] = np.arange(len(faces))
        targets = faces[best_face]
    if len(ignore):
        # Small faces can be narrower than a grid cell: widen each ignore region by half the anchor's stride,
        # otherwise no anchor centre lands inside it and the face would be taught as background.
        c, pad = ANCHORS[:, :2], HALF_STRIDE[:, None]
        inside = ((c[:, None, 0] >= ignore[None, :, 0] - pad) & (c[:, None, 0] <= ignore[None, :, 2] + pad) &
                  (c[:, None, 1] >= ignore[None, :, 1] - pad) & (c[:, None, 1] <= ignore[None, :, 3] + pad)).any(1)
        labels[inside & (labels == 0)] = -1
    return labels, targets


def make_batch(idx, seed):
    rng = np.random.default_rng(seed)
    xs, ls, ts = [], [], []
    for i in idx:
        img, boxes = train_cache[i]
        x, faces, cut = augment(img, boxes, rng)
        lab, tgt = encode(faces, cut)
        xs.append(x); ls.append(lab); ts.append(tgt)
    return np.stack(xs), np.stack(ls), np.stack(ts)


def eval_targets(ev):
    out = [encode(to_tensor_space(g, lb), np.zeros((0, 4))) for g, lb in zip(ev['gt'], ev['lb'])]
    return np.stack([o[0] for o in out]), np.stack([o[1] for o in out])


val_set['labels'], val_set['targets'] = eval_targets(val_set)
_lab = val_set['labels']
print(f'val targets: {(_lab == 1).sum(1).mean():.1f} face anchors / image, '
      f'{(_lab == -1).sum(1).mean():.1f} ignored, {(_lab == 0).sum(1).mean():.0f} background')

## 5. Loss

* **Classification** - binary cross-entropy on face anchors plus the hardest background anchors (3 per face
  anchor, at least 16 per image): classic SSD hard-negative mining. Plain cross-entropy keeps the scores
  calibrated, which matters because MediaPipe cuts at a fixed confidence (0.5).
* **Box** - GIoU loss between the decoded box and the face, on face anchors (x2).
* **Landmarks** - WIDER FACE / FDDB have no facial keypoints, so on every anchor where the original model sees a
  face, the 6 keypoints are pulled towards the frozen original's prediction (x20). Without this they would
  drift, and MediaPipe would return meaningless keypoints.

The history logs each term already multiplied by its weight, so the three curves add up to the total.

In [ ]:
BOX_WEIGHT, KP_WEIGHT = 2.0, 20.0
NEG_POS_RATIO, MIN_NEG = 3, 16
ANCHOR_XY = tf.constant(ANCHORS[:, :2])


def decode_tf(reg):
    r = reg / INPUT
    c = ANCHOR_XY + r[..., :2]
    return tf.concat([c - r[..., 2:4] / 2, c + r[..., 2:4] / 2], -1)


def giou(a, b):
    ix1, iy1 = tf.maximum(a[..., 0], b[..., 0]), tf.maximum(a[..., 1], b[..., 1])
    ix2, iy2 = tf.minimum(a[..., 2], b[..., 2]), tf.minimum(a[..., 3], b[..., 3])
    inter = tf.nn.relu(ix2 - ix1) * tf.nn.relu(iy2 - iy1)
    area = lambda x: tf.nn.relu(x[..., 2] - x[..., 0]) * tf.nn.relu(x[..., 3] - x[..., 1])
    union = area(a) + area(b) - inter + 1e-9
    enclose = ((tf.maximum(a[..., 2], b[..., 2]) - tf.minimum(a[..., 0], b[..., 0])) *
               (tf.maximum(a[..., 3], b[..., 3]) - tf.minimum(a[..., 1], b[..., 1])) + 1e-9)
    return inter / union - (enclose - union) / enclose


def compute_losses(reg, cls, labels, targets, t_reg, t_cls):
    logits = cls[..., 0]
    pos = tf.cast(labels > 0.5, tf.float32)
    neg = tf.cast(tf.equal(labels, 0.0), tf.float32)
    n_pos = tf.reduce_sum(pos, 1)
    bce = tf.nn.sigmoid_cross_entropy_with_logits(labels=pos, logits=logits)
    neg_loss = bce * neg
    k = tf.cast(tf.minimum(tf.maximum(NEG_POS_RATIO * n_pos, MIN_NEG), tf.reduce_sum(neg, 1)), tf.int32)
    thresh = tf.gather(tf.sort(neg_loss, axis=1, direction='DESCENDING'), tf.maximum(k - 1, 0), batch_dims=1)
    hard_neg = neg * tf.cast(neg_loss >= thresh[:, None], tf.float32) * tf.cast(k[:, None] > 0, tf.float32)
    total_pos = tf.maximum(tf.reduce_sum(n_pos), 1.0)
    cls_loss = tf.reduce_sum(bce * (pos + hard_neg)) / total_pos

    box_loss = BOX_WEIGHT * tf.reduce_sum((1.0 - giou(decode_tf(reg), targets)) * pos) / total_pos

    teacher_face = tf.cast(tf.sigmoid(t_cls[..., 0]) > 0.5, tf.float32)
    kp_err = tf.reduce_mean(tf.abs(reg[..., 4:] - t_reg[..., 4:]), -1) / INPUT
    kp_loss = KP_WEIGHT * tf.reduce_sum(kp_err * teacher_face) / tf.maximum(tf.reduce_sum(teacher_face), 1.0)
    return cls_loss + box_loss + kp_loss, cls_loss, box_loss, kp_loss

## 6. Metrics - identical to the YOLO tables

`evaluate` reproduces Ultralytics 8.4's validator line by line (`match_predictions`, `ap_per_class`, 101-point
COCO AP; checked to give the same numbers as Ultralytics' own functions), so every column means the same thing
for BlazeFace as for YOLOv8n:

* **Precision** = TP / (TP + FP), **Recall** = TP / (TP + FN), **F1** = 2PR / (P + R), all at the confidence
  that maximises F1 (as Ultralytics reports them);
* **Accuracy** = TP / (TP + FP + FN) (single-class detection has no true negatives) - the same quantity the
  paper's "Accuracy" column uses;
* **mAP@0.5** and **mAP@0.5:0.95** - area under the precision-recall curve at IoU 0.5, and averaged over IoU
  0.50-0.95.

Detections are scored down to confidence 0.01 for the PR curve, as Ultralytics does. The `deploy_*` columns are
measured at MediaPipe's default 0.5 instead - what the live system actually outputs.

In [ ]:
IOU_T = np.linspace(0.5, 0.95, 10)


def match_predictions(gt, det):
    """TP matrix (n_det, 10) - same algorithm as ultralytics BaseValidator.match_predictions."""
    tp = np.zeros((len(det), len(IOU_T)), bool)
    if len(gt) == 0 or len(det) == 0:
        return tp
    iou = iou_matrix(gt, det)
    for i, t in enumerate(IOU_T):
        m = np.nonzero(iou >= t)
        if m[0].size:
            mm = np.stack([m[0], m[1], iou[m]], 1)
            if m[0].size > 1:
                mm = mm[mm[:, 2].argsort()[::-1]]
                mm = mm[np.unique(mm[:, 1], return_index=True)[1]]
                mm = mm[np.unique(mm[:, 0], return_index=True)[1]]
            tp[mm[:, 1].astype(int), i] = True
    return tp


def smooth(y, f=0.1):
    nf = round(len(y) * f * 2) // 2 + 1
    p = np.ones(nf // 2)
    return np.convolve(np.concatenate((p * y[0], y, p * y[-1])), np.ones(nf) / nf, mode='valid')


def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [recall[-1] if len(recall) else 1.0], [1.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(np.concatenate(([1.0], precision, [0.0], [0.0])))))
    x = np.linspace(0, 1, 101)
    return trapezoid(np.interp(x, mrec, mpre), x)


def face_size(gt, lb):
    """Face size as a fraction of the image's longer side (= of the 128-px input)."""
    t = to_tensor_space(gt, lb)
    return np.sqrt(np.clip(t[:, 2] - t[:, 0], 0, None) * np.clip(t[:, 3] - t[:, 1], 0, None))


def evaluate(dets, gts, lbs, min_face=0.0):
    """min_face > 0: faces smaller than that fraction of the image are 'ignore' regions (neither TP nor FN,
    and detections on them are not FP)."""
    tps, confs, n_gt = [], [], 0
    for (boxes, scores), g, lb in zip(dets, gts, lbs):
        if min_face > 0 and len(g):
            size = face_size(g, lb)
            small, g = g[size < min_face], g[size >= min_face]
            if len(small) and len(boxes):
                hit = match_predictions(g, boxes)[:, 0] if len(g) else np.zeros(len(boxes), bool)
                keep = ~((iou_matrix(boxes, small).max(1) >= 0.5) & ~hit)
                boxes, scores = boxes[keep], scores[keep]
        n_gt += len(g)
        tps.append(match_predictions(g, boxes)); confs.append(scores)
    tp, conf = np.concatenate(tps), np.concatenate(confs)
    order = np.argsort(-conf)
    tp, conf = tp[order], conf[order]
    tpc, fpc = tp.cumsum(0), (1 - tp).cumsum(0)
    recall, precision = tpc / (n_gt + 1e-16), tpc / (tpc + fpc + 1e-16)
    ap = np.array([compute_ap(recall[:, j], precision[:, j]) for j in range(10)]) if len(tp) else np.zeros(10)
    px = np.linspace(0, 1, 1000)
    p_curve = np.interp(-px, -conf, precision[:, 0], left=1) if len(tp) else np.ones(1000)
    r_curve = np.interp(-px, -conf, recall[:, 0], left=0) if len(tp) else np.zeros(1000)
    i = smooth(2 * p_curve * r_curve / (p_curve + r_curve + 1e-16)).argmax()
    p, r = p_curve[i], r_curve[i]
    tp_n = round(r * n_gt)
    fp_n = round(tp_n / (p + 1e-16) - tp_n)
    d = conf >= DEPLOY_CONF
    tp_d = int(tp[d, 0].sum())
    return dict(precision=p, recall=r, f1=2 * p * r / (p + r + 1e-16), accuracy=p * r / (p + r - p * r + 1e-16),
                map50=ap[0], map=ap.mean(), best_conf=px[i], TP=tp_n, FP=fp_n, FN=n_gt - tp_n, faces=n_gt,
                deploy_precision=tp_d / max(int(d.sum()), 1), deploy_recall=tp_d / max(n_gt, 1),
                deploy_TP=tp_d, deploy_FP=int(d.sum()) - tp_d, deploy_FN=n_gt - tp_d)


def score(model, ev, min_faces=(0.0,)):
    dets = detect(*predict(model, ev['x']), ev['lb'])
    return [evaluate(dets, ev['gt'], ev['lb'], m) for m in min_faces]

## 7. Train

Adam, 3-epoch warm-up then cosine decay, batch 32, gradient clipping. After every epoch the notebook records the
training losses, the validation loss, validation metrics (all faces and webcam-range faces) and the same metrics
on 800 fixed, un-augmented training images (Keras-style "training accuracy"). The epoch with the best validation
mAP@0.5:0.95 is kept as `best.weights.h5` (the same rule Ultralytics uses for `best.pt`); training stops early
after 20 epochs without improvement. **Epoch 0 is the original, un-tuned Google model** - the baseline.

Everything is written to Drive after each epoch, so a disconnected session resumes on the next Run all.

In [ ]:
student = blazeface_from_tflite(BASE_TFLITE)
steps = math.ceil(len(train_cache) / BATCH)
warmup = min(3 * steps, EPOCHS * steps // 10)
schedule = keras.optimizers.schedules.CosineDecay(LR * 0.1, EPOCHS * steps - warmup, alpha=0.01,
                                                  warmup_target=LR, warmup_steps=warmup)
optimizer = keras.optimizers.Adam(schedule)
optimizer.build(student.trainable_variables)


@tf.function(reduce_retracing=True)
def train_step(x_uint8, labels, targets):
    x = tf.cast(x_uint8, tf.float32) / 127.5 - 1.0
    t_reg, t_cls = teacher(x, training=False)
    with tf.GradientTape() as tape:
        reg, cls = student(x, training=True)
        losses = compute_losses(reg, cls, labels, targets, t_reg, t_cls)
    grads = tape.gradient(losses[0], student.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 10.0)
    optimizer.apply_gradients(zip(grads, student.trainable_variables))
    return losses


@tf.function(reduce_retracing=True)
def loss_step(x_uint8, labels, targets):
    x = tf.cast(x_uint8, tf.float32) / 127.5 - 1.0
    t_reg, t_cls = teacher(x, training=False)
    reg, cls = student(x, training=False)
    return compute_losses(reg, cls, labels, targets, t_reg, t_cls)


def validation_loss():
    total, n = np.zeros(4), 0
    for i in range(0, len(val_set['x']), 128):
        out = loss_step(val_set['x'][i:i + 128], val_set['labels'][i:i + 128], val_set['targets'][i:i + 128])
        total += np.array([float(v) for v in out]) * len(val_set['x'][i:i + 128])
        n += len(val_set['x'][i:i + 128])
    return total / n


def epoch_row(epoch, train_losses, lr):
    vl = validation_loss()
    v_all, v_in = score(student, val_set, (0.0, IN_RANGE_MIN_FACE))
    (t_all,) = score(student, train_eval_set)
    row = dict(epoch=epoch, lr=lr)
    row.update(zip(['train_loss', 'train_cls_loss', 'train_box_loss', 'train_kp_loss'], train_losses))
    row.update(zip(['val_loss', 'val_cls_loss', 'val_box_loss', 'val_kp_loss'], vl))
    row.update({f'val_{k}': v for k, v in v_all.items()})
    row.update({f'valrange_{k}': v for k, v in v_in.items()})
    row.update({f'train_{k}': v for k, v in t_all.items()})
    return row


run_dir = Path(OUT_DIR, 'blazeface')
run_dir.mkdir(parents=True, exist_ok=True)
hist_path, state_path = run_dir / 'history.csv', run_dir / 'state.json'
if state_path.exists():   # resume an interrupted run
    state = json.loads(state_path.read_text())
    history = pd.read_csv(hist_path).to_dict('records')
    student.load_weights(str(run_dir / 'last.weights.h5'))
    optimizer.iterations.assign(state['epoch'] * steps)
    print(f'resuming after epoch {state["epoch"]} (best so far: epoch {state["best_epoch"]})')
else:
    state = dict(epoch=0, best_epoch=0, best_fitness=-1.0, done=False)
    history = [epoch_row(0, [math.nan] * 4, 0.0)]
    state['best_fitness'] = history[0]['val_map']
    student.save_weights(str(run_dir / 'best.weights.h5'))


def log(row, secs):
    print(f"epoch {row['epoch']:3d} | loss {row['train_loss']:.3f} val {row['val_loss']:.3f} | "
          f"VAL P {row['val_precision']:.3f} R {row['val_recall']:.3f} mAP50 {row['val_map50']:.3f} "
          f"mAP50-95 {row['val_map']:.3f} | webcam-range mAP50 {row['valrange_map50']:.3f} | "
          f"train mAP50 {row['train_map50']:.3f} | {secs:.0f}s")


if state['epoch'] == 0:
    log(history[0], 0)
with ThreadPoolExecutor(4) as pool:
    for epoch in range(state['epoch'] + 1, EPOCHS + 1):
        if state['done']:
            break
        t0 = time.time()
        order = np.random.default_rng(SEED + epoch).permutation(len(train_cache))
        batches = pool.map(make_batch, [order[s * BATCH:(s + 1) * BATCH] for s in range(steps)],
                           [SEED * 100_000 + epoch * 1_000 + s for s in range(steps)])
        sums = np.zeros(4)
        for x, labels, targets in batches:
            sums += np.array([float(v) for v in train_step(x, labels, targets)])
        row = epoch_row(epoch, sums / steps, float(schedule(optimizer.iterations)))
        history.append(row)
        student.save_weights(str(run_dir / 'last.weights.h5'))
        if row['val_map'] > state['best_fitness']:
            state.update(best_fitness=row['val_map'], best_epoch=epoch)
            student.save_weights(str(run_dir / 'best.weights.h5'))
        state['epoch'] = epoch
        if epoch - state['best_epoch'] >= PATIENCE:
            state['done'] = True
            print(f'early stop: no val mAP@0.5:0.95 improvement for {PATIENCE} epochs')
        pd.DataFrame(history).to_csv(hist_path, index=False)
        state_path.write_text(json.dumps(state))
        log(row, time.time() - t0)

print(f"best epoch: {state['best_epoch']} (val mAP@0.5:0.95 = {state['best_fitness'] * 100:.2f}%)")

## 8. Final evaluation - TEST split (run once)

Scores, on the 402 test images, both **the original Google model** (baseline) and **the fine-tuned model**
(`best.weights.h5`), with exactly the same code:

* **all faces** - every annotated face counts, the same protocol as the YOLO table;
* **webcam-range faces** - faces at least 10% of the image's longer side (an examinee in front of a webcam is
  typically 20-40%); smaller faces are ignore regions.

It also records the mean confidence of detections at conf >= 0.25 (the paper's "Mean Confidence" column) and the
TP / FP / FN counts at MediaPipe's default threshold 0.5 (what the live system outputs).

In [ ]:
best = blazeface_from_tflite(BASE_TFLITE)
best.load_weights(str(run_dir / 'best.weights.h5'))
history_df = pd.DataFrame(history)
best_epoch = int(state['best_epoch'])
test_set = load_eval_set(list_split('test'))
N_TEST = len(test_set['x'])

BF_MODELS = {'blazeface-pretrained': teacher, 'blazeface': best}
DISPLAY = {'yolov8n': 'YOLOv8n', 'yolo11n': 'YOLOv11n', 'rtdetr-l': 'RT-DETR-l',
           'blazeface-pretrained': 'BlazeFace (original)', 'blazeface': 'BlazeFace (fine-tuned)'}
TYPE = {'yolov8n': '1-stage CNN', 'yolo11n': '1-stage CNN', 'rtdetr-l': 'Transformer',
        'blazeface-pretrained': '1-stage CNN', 'blazeface': '1-stage CNN'}

test_dets, test_rows = {}, []
for name, model in BF_MODELS.items():
    reg, cls = predict(model, test_set['x'])
    test_dets[name] = detect(reg, cls, test_set['lb'])
    confident = [s for _, s in detect(reg, cls, test_set['lb'], min_score=0.25)]
    for protocol, min_face in (('all faces', 0.0), ('webcam-range faces', IN_RANGE_MIN_FACE)):
        m = evaluate(test_dets[name], test_set['gt'], test_set['lb'], min_face)
        row = {'model': name, 'protocol': protocol,
               'mAP50 (%)': m['map50'] * 100, 'mAP50-95 (%)': m['map'] * 100,
               'Precision (%)': m['precision'] * 100, 'Recall (%)': m['recall'] * 100,
               'F1 (%)': m['f1'] * 100, 'Accuracy (%)': m['accuracy'] * 100,
               'TP': m['TP'], 'FP': m['FP'], 'FN': m['FN'], 'Faces': m['faces'],
               'Precision @0.5 (%)': m['deploy_precision'] * 100, 'Recall @0.5 (%)': m['deploy_recall'] * 100,
               'TP @0.5': m['deploy_TP'], 'FP @0.5': m['deploy_FP'], 'FN @0.5': m['deploy_FN']}
        if protocol == 'all faces':
            all_conf = np.concatenate(confident)
            row['Mean Confidence (%)'] = float(all_conf.mean() * 100) if len(all_conf) else 0.0
            row['Detections (n)'] = int(len(all_conf))
        test_rows.append(row)

bf_test_df = pd.DataFrame(test_rows).round(2).astype({'Detections (n)': 'Int64'})
bf_test_df.to_csv(f'{OUT_DIR}/test_results_blazeface_70_20_10.csv', index=False)
print('TEST split - report these:')
bf_test_df

## 9. Export for MediaPipe + speed

* `blaze_face_short_range_finetuned.tflite` - drop-in replacement for
  `frontend/public/models/blaze_face_short_range.tflite` (browser overlay).
* `face_landmarker_finetuned.task` - Google's `face_landmarker.task` with the fine-tuned detector inside; drop-in
  replacement for `backend/ai/models/face_landmarker.task` (server pipeline).

Both are loaded back into MediaPipe here to prove they work. Speed is measured for the exported `.tflite` on
this runtime's CPU (how it runs in the browser / on the server) and, if there is one, on the GPU at batch 16
(how Ultralytics measured the YOLO models' T4 numbers).

In [ ]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):   # the converter prints every captured variable
    FT_TFLITE = export_tflite(best, f'{OUT_DIR}/blaze_face_short_range_finetuned.tflite')
FT_TASK = repack_face_landmarker(BASE_TASK, FT_TFLITE, f'{OUT_DIR}/face_landmarker_finetuned.task')
print(f'exported {FT_TFLITE} ({os.path.getsize(FT_TFLITE) / 1024:.0f} KB)')

n_mp, n_ours, dscore, miou = compare_with_mediapipe(best, FT_TFLITE, _val_paths)
print(f'MediaPipe FaceDetector + fine-tuned .tflite vs Keras model: {n_mp} vs {n_ours} faces, '
      f'mean |score diff| {dscore:.4f}, mean IoU {miou:.3f}')
assert abs(n_mp - n_ours) <= max(1, n_ours // 20) and miou > 0.9, 'exported model does not match - do not ship it'

def face_mesh_found(task_path):
    lm = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=task_path), num_faces=1))
    n = sum(bool(lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=read_rgb(p))).face_landmarks)
            for p in _val_paths)
    lm.close()
    return n


print(f'FaceLandmarker - face mesh found on {len(_val_paths)} val images: original task '
      f'{face_mesh_found(BASE_TASK)}, fine-tuned task {face_mesh_found(FT_TASK)}')


def tflite_ms(path, n=300):
    it = Interpreter(path, num_threads=os.cpu_count())
    it.allocate_tensors()
    idx = it.get_input_details()[0]['index']
    xs = normalize(test_set['x'][:50])
    for i in range(20):
        it.set_tensor(idx, xs[i % 50][None]); it.invoke()
    t = time.perf_counter()
    for i in range(n):
        it.set_tensor(idx, xs[i % 50][None]); it.invoke()
    return (time.perf_counter() - t) / n * 1000


def keras_gpu_ms(model, batch=16, reps=50):
    if not tf.config.list_physical_devices('GPU'):
        return math.nan
    x = tf.constant(normalize(test_set['x'][:batch]))
    f = tf.function(lambda x: model(x, training=False))
    for _ in range(5):
        f(x)[0].numpy()
    t = time.perf_counter()
    for _ in range(reps):
        f(x)[0].numpy()
    return (time.perf_counter() - t) / reps / batch * 1000


CPU_NAME = (subprocess.run("lscpu | sed -n 's/^Model name: *//p'", shell=True, capture_output=True, text=True)
            .stdout.strip() or 'CPU') + f' ({os.cpu_count()} threads)'
speed = {}
for name, path in (('blazeface-pretrained', BASE_TFLITE), ('blazeface', FT_TFLITE)):
    ms = tflite_ms(path)
    speed[name] = {'Inference CPU (ms)': ms, 'FPS CPU': 1000 / ms,
                   'Inference GPU (ms)': keras_gpu_ms(BF_MODELS[name]), 'Params (M)': best.count_params() / 1e6}
    speed[name]['FPS GPU'] = 1000 / speed[name]['Inference GPU (ms)']
print(f'CPU: {CPU_NAME}')
pd.DataFrame(speed).T.round(3)

## 10. Comparison with the YOLO models (optional)

If `train_eval_face_70_20_10.ipynb` has been run, its results are in `MyDrive/knowing-eye-70-20-10/`. This cell
puts them in the same table:

* accuracy columns for YOLO come from its own `test_results_70_20_10.csv` (Ultralytics validator, same test
  split, same metric definitions);
* if the YOLO `best.pt` weights are there too, it also scores them on **webcam-range faces** with the code above,
  and times them on **this runtime's CPU** at their native 640x640 input - so the speed chart compares every model
  on the same hardware.

Without the YOLO outputs it simply compares the original and fine-tuned BlazeFace.

In [ ]:
YOLO_MODELS = ['yolov8n', 'yolo11n', 'rtdetr-l']
yolo_csv = Path(YOLO_DIR, 'test_results_70_20_10.csv')
yolo_test = pd.read_csv(yolo_csv).set_index('model') if yolo_csv.exists() else pd.DataFrame()
yolo_weights = {n: Path(YOLO_DIR, n, 'best.pt') for n in YOLO_MODELS if Path(YOLO_DIR, n, 'best.pt').exists()}
print('YOLO results:', list(yolo_test.index) or 'none', '| YOLO weights:', list(yolo_weights) or 'none')

yolo_range = {}
if yolo_weights:
    import importlib.util
    if importlib.util.find_spec('ultralytics') is None:   # same version as the YOLO notebook
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.155'], check=True)
    from ultralytics import RTDETR, YOLO
    test_paths = test_set['paths']
    letterboxes = test_set['lb']
    device = 0 if tf.config.list_physical_devices('GPU') else 'cpu'
    for name, weights in yolo_weights.items():
        model = (RTDETR if name.startswith('rtdetr') else YOLO)(str(weights))
        dets = []
        for i in range(0, N_TEST, 16):   # Ultralytics validation settings: conf 0.001, NMS IoU 0.7
            for r in model.predict(test_paths[i:i + 16], imgsz=640, conf=0.001, iou=0.7, max_det=300,
                                   device=device, verbose=False):
                dets.append((r.boxes.xyxyn.cpu().numpy(), r.boxes.conf.cpu().numpy()))
        all_m = evaluate(dets, test_set['gt'], letterboxes)
        rng_m = evaluate(dets, test_set['gt'], letterboxes, IN_RANGE_MIN_FACE)
        yolo_range[name] = (all_m, rng_m)
        cpu = (RTDETR if name.startswith('rtdetr') else YOLO)(str(weights))
        cpu.predict(test_paths[:3], imgsz=640, device='cpu', verbose=False)   # warm-up
        ms = np.mean([r.speed['inference'] for p in test_paths[:30]
                      for r in cpu.predict(p, imgsz=640, device='cpu', verbose=False)])
        cpu.fuse()   # count parameters the way Ultralytics reports them (Conv+BN fused)
        speed[name] = {'Inference CPU (ms)': ms, 'FPS CPU': 1000 / ms,
                       'Params (M)': sum(x.numel() for x in cpu.model.parameters()) / 1e6}
        # model.val() batches images into rectangles (rect=True); predict() letterboxes each image, which is
        # what val(rect=False) does - with that setting the two give identical numbers.
        print(f'{DISPLAY[name]}: all-faces mAP50 {all_m["map50"] * 100:.2f}% (this notebook, = val(rect=False)) vs '
              f'{yolo_test.loc[name, "mAP50 (%)"] if name in yolo_test.index else float("nan"):.2f}% (YOLO notebook, '
              f'val(rect=True)) | webcam-range mAP50 {rng_m["map50"] * 100:.2f}% | CPU {ms:.1f} ms')

In [ ]:
# One comparison table, every model, TEST split - the paper's Table 2 layout.
def comparison_rows(protocol):
    rows = []
    for name in YOLO_MODELS:
        if protocol == 'all faces' and name in yolo_test.index:
            y = yolo_test.loc[name]
            r = {k: y[k] for k in ('mAP50 (%)', 'mAP50-95 (%)', 'Precision (%)', 'Recall (%)', 'F1 (%)',
                                   'Accuracy (%)', 'Mean Confidence (%)') if k in y}
            r['Inference GPU (ms)'], r['FPS GPU'] = y.get('Inference T4 (ms)'), y.get('FPS T4')
            r['Params (M)'] = y.get('Params (M)')
        elif name in yolo_range:
            m = yolo_range[name][0 if protocol == 'all faces' else 1]
            r = {'mAP50 (%)': m['map50'] * 100, 'mAP50-95 (%)': m['map'] * 100,
                 'Precision (%)': m['precision'] * 100, 'Recall (%)': m['recall'] * 100,
                 'F1 (%)': m['f1'] * 100, 'Accuracy (%)': m['accuracy'] * 100}
        else:
            continue
        if name in yolo_test.index and 'Params (M)' in yolo_test:
            r['Params (M)'] = yolo_test.loc[name, 'Params (M)']
        r.update({k: v for k, v in speed.get(name, {}).items() if k not in r or pd.isna(r[k])})
        rows.append({'model': name, **r})
    for name in BF_MODELS:
        t = bf_test_df[(bf_test_df.model == name) & (bf_test_df.protocol == protocol)].iloc[0]
        r = {k: t[k] for k in ('mAP50 (%)', 'mAP50-95 (%)', 'Precision (%)', 'Recall (%)', 'F1 (%)',
                               'Accuracy (%)', 'Mean Confidence (%)') if k in t and not pd.isna(t[k])}
        rows.append({'model': name, **r, **speed[name]})
    df = pd.DataFrame(rows)
    df.insert(1, 'Type', df['model'].map(TYPE))
    df['model'] = df['model'].map(DISPLAY)
    return df.round(2)


compare_all = comparison_rows('all faces')
compare_range = comparison_rows('webcam-range faces')
compare_all.to_csv(f'{OUT_DIR}/comparison_all_faces.csv', index=False)
compare_range.to_csv(f'{OUT_DIR}/comparison_webcam_range_faces.csv', index=False)
print('All faces (paper Table 2 protocol):')
display(compare_all)
print(f'Webcam-range faces (>= {IN_RANGE_MIN_FACE:.0%} of the image side):')
display(compare_range)

## 11. Tables and figures for the paper

Same layout, palette and look as the paper's existing face-detection figures (Figures 47-55). All saved as
200-dpi PNGs in `MyDrive/knowing-eye-blazeface-70-20-10/figures/`.

| Paper item (YOLOv8n version) | BlazeFace file |
|---|---|
| Table 1 / Table 3 - metrics at epochs 20, 40, 60, 80, 100 + average | `blazeface/history_checkpoints.csv` (printed below) |
| Fig. 47 - performance metrics as a function of training epochs | `blazeface_checkpoint_metrics.png` |
| Fig. 48 - final detection performance (bar chart) | `blazeface_test_metrics_bar.png` |
| Table 2 - algorithms on the same dataset, best per column highlighted | `compare_table.png` (+ CSV) |
| Fig. 48 (comparison) - accuracy metrics across models | `compare_test_metrics.png` |
| Fig. 49 - accuracy vs speed, bubble = model size | `compare_accuracy_vs_speed.png` |
| Fig. 50 - validation P / R / mAP@0.5 / mAP@0.5:0.95 across all epochs | `blazeface_val_metrics_all_epochs.png` |
| Fig. 51 - training loss components across all epochs | `blazeface_loss_components.png` |

Extra figures: Keras-style training-vs-validation accuracy and loss, training-vs-validation metrics, the test
confusion matrix, original-vs-fine-tuned on both protocols, and recall by face size (the clearest picture of
what fine-tuning changed and where a 128-px detector stops seeing faces).

**Reading the curves.** Epoch 0 is the original Google model. Training loss is measured on augmented batches,
validation loss on clean images, so validation loss can sit below training loss. "Training" accuracy/metrics are
measured on 800 fixed un-augmented training images with exactly the validation code, so the train-vs-val gap is
a fair overfitting check. The dotted line marks the epoch kept as the final model. Checkpoint tables/charts are
**validation** numbers; bar charts, the comparison and the confusion matrix are **test** numbers.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Palette and look matched to the paper's existing figures.
METRIC_COLORS = {'mAP@0.5': '#1F4E8C', 'Precision': '#2E8B57', 'F1-Score': '#D98C00', 'Recall': '#C0504D',
                 'mAP@0.5:0.95': '#6A3D9A', 'Accuracy': '#2B8C8C'}
MODEL_COLORS = {'YOLOv8n': '#1F4E8C', 'YOLOv11n': '#2E8B57', 'RT-DETR-l': '#D98C00',
                'BlazeFace (original)': '#8C8C8C', 'BlazeFace (fine-tuned)': '#C0504D'}
TRAIN_C, VAL_C = '#1F4E8C', '#D98C00'
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11, 'axes.titleweight': 'bold', 'axes.titlesize': 13,
    'axes.labelweight': 'bold', 'axes.labelsize': 12, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
})


def style(ax, xlabel=None, ylabel=None, title=None):
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_linewidth(1.6)
    ax.grid(axis='y', color='#DDDDDD', linewidth=1)
    ax.set_axisbelow(True)
    if xlabel:
        ax.set_xlabel(xlabel)
        if 'Epoch' in xlabel:
            ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if ylabel:
        ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)


def legend_top(ax, ncol):
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=ncol, frameon=False,
              prop={'weight': 'bold', 'size': 10})


def mark_best(ax, epoch=None):
    epoch = best_epoch if epoch is None else epoch
    ax.axvline(epoch, color='#5B7A99', linestyle=':', linewidth=1.5)
    ax.annotate(f'best: epoch {epoch}', xy=(epoch, 1), xycoords=('data', 'axes fraction'), xytext=(0, 4),
                textcoords='offset points', ha='center', va='bottom', color='#1F4E8C', fontsize=9,
                fontweight='bold')


def save(fig, fname):
    fig.savefig(f'{FIG_DIR}/{fname}', dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


h = history_df.copy()
for col in [c for c in h.columns if c.startswith(('val_', 'valrange_', 'train_'))
            and c.split('_', 1)[1] in ('precision', 'recall', 'f1', 'accuracy', 'map50', 'map')]:
    h[col] = h[col] * 100

In [ ]:
# Paper Table 1 / Table 3 layout: VALIDATION metrics at checkpoint epochs + their average.
CHECKPOINTS = [e for e in (20, 40, 60, 80, 100) if e in set(h['epoch'])] or list(h['epoch'][1:])
cols = {'epoch': 'Epoch', 'val_accuracy': 'Accuracy (%)', 'val_recall': 'Recall (%)',
        'val_precision': 'Precision (%)', 'val_f1': 'F1-Score (%)', 'val_map50': 'mAP@0.5 (%)'}
table1 = h[h['epoch'].isin(CHECKPOINTS)][list(cols)].rename(columns=cols)
table1 = pd.concat([table1, table1.drop(columns='Epoch').mean().to_frame().T.assign(Epoch='Average')])
table1 = table1.round(2).reset_index(drop=True)
table1.to_csv(f'{run_dir}/history_checkpoints.csv', index=False)

# Extended version: also the original model (epoch 0), the kept epoch, the last epoch, mAP@0.5:0.95,
# webcam-range mAP and training accuracy / losses.
ext_cols = {**cols, 'val_map': 'mAP@0.5:0.95 (%)', 'valrange_map50': 'Webcam-range mAP@0.5 (%)',
            'train_accuracy': 'Train Accuracy (%)', 'train_loss': 'Train Loss', 'val_loss': 'Val Loss'}
keep = sorted({0, best_epoch, int(h['epoch'].max()), *CHECKPOINTS})
table_ext = h[h['epoch'].isin(keep)][list(ext_cols)].rename(columns=ext_cols).round(2)
table_ext['Note'] = table_ext['Epoch'].map(lambda e: 'original model' if e == 0 else
                                           'kept (best val mAP@0.5:0.95)' if e == best_epoch else '')
table_ext.to_csv(f'{run_dir}/history_checkpoints_extended.csv', index=False)
h.round(4).to_csv(f'{run_dir}/history_all_epochs.csv', index=False)
print(f'BlazeFace (fine-tuned) - validation metrics at checkpoint epochs (paper Table 1 / Table 3 layout)')
display(table1)
display(table_ext)

In [ ]:
# ---------- Paper Fig. 47: metrics as a function of training epochs (checkpoints) ----------
def spread(values, min_gap):
    """Label positions near `values`, at least `min_gap` apart (keeps end labels from overlapping)."""
    order = np.argsort(values)
    pos = np.array(values, float)[order]
    for i in range(1, len(pos)):
        pos[i] = max(pos[i], pos[i - 1] + min_gap)
    pos -= max(0.0, (pos[-1] - max(values)) / 2)
    out = np.empty_like(pos)
    out[order] = pos
    return out


pts = h[h['epoch'].isin(CHECKPOINTS)]
series = [('val_map50', 'mAP@0.5'), ('val_precision', 'Precision'), ('val_f1', 'F1-Score'), ('val_recall', 'Recall')]
fig, ax = plt.subplots(figsize=(13, 6))
for key, label in series:
    ax.plot(pts['epoch'], pts[key], color=METRIC_COLORS[label], lw=2.5, marker='o', ms=7, label=label)
style(ax, None, 'Score (%)')
ends = [pts[k].iloc[-1] for k, _ in series]
lo, hi = ax.get_ylim()
for (key, label), y_end, y_text in zip(series, ends, spread(ends, (hi - lo) * 0.045)):
    ax.annotate(f'{y_end:.2f}', (pts['epoch'].iloc[-1], y_end), xytext=(8, y_text),
                textcoords=('offset points', 'data'), va='center', color=METRIC_COLORS[label], fontweight='bold')
ax.set_xticks(pts['epoch'])
legend_top(ax, ncol=4)
save(fig, 'blazeface_checkpoint_metrics.png')

# ---------- Paper Fig. 48: final detection performance (TEST) ----------
t = bf_test_df[(bf_test_df.model == 'blazeface') & (bf_test_df.protocol == 'all faces')].iloc[0]
items = [('mAP@0.5', t['mAP50 (%)']), ('Recall', t['Recall (%)']), ('Precision', t['Precision (%)']),
         ('F1-Score', t['F1 (%)'])]
fig, ax = plt.subplots(figsize=(13, 5.5))
for i, (label, v) in enumerate(items):
    c = METRIC_COLORS[label]
    ax.bar(i, v, width=0.45, color=c, edgecolor='black', linewidth=1.2)
    ax.text(i, v + 1.5, f'{v:.2f}', ha='center', va='bottom', color=c, fontweight='bold', fontsize=12)
ax.set_xticks(range(len(items)), [l for l, _ in items], fontweight='bold')
style(ax, None, 'Score (%)')
ax.set_ylim(0, 105)
save(fig, 'blazeface_test_metrics_bar.png')

# ---------- Paper Fig. 50: validation metrics across all epochs ----------
fig, ax = plt.subplots(figsize=(13, 5.5))
for key, label in [('val_map50', 'mAP@0.5'), ('val_map', 'mAP@0.5:0.95'), ('val_precision', 'Precision'),
                   ('val_recall', 'Recall')]:
    ax.plot(h['epoch'], h[key], color=METRIC_COLORS[label], lw=2.2, label=label)
style(ax, 'Training Epoch', 'Score (%)')
ax.set_ylim(0, 100)
ax.set_xlim(0, h['epoch'].max())
mark_best(ax)
legend_top(ax, ncol=4)
save(fig, 'blazeface_val_metrics_all_epochs.png')

# ---------- Paper Fig. 51: training loss components across all epochs ----------
fig, ax = plt.subplots(figsize=(13, 5.5))
for key, label, c in [('train_cls_loss', 'Class Loss', '#C0504D'), ('train_box_loss', 'Box (GIoU) Loss', '#D98C00'),
                      ('train_kp_loss', 'Landmark Loss', '#6A3D9A')]:
    ax.plot(h['epoch'], h[key], color=c, lw=2.2, label=label)
style(ax, 'Training Epoch', 'Loss')
ax.set_ylim(bottom=0)
ax.set_xlim(1, h['epoch'].max())
legend_top(ax, ncol=3)
save(fig, 'blazeface_loss_components.png')

In [ ]:
# ---------- Extra: Keras-style model accuracy / model loss, training vs validation ----------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.plot(h['epoch'], h['train_accuracy'], color=TRAIN_C, lw=2.2, label='Training')
a1.plot(h['epoch'], h['val_accuracy'], color=VAL_C, lw=2.2, label='Validation')
style(a1, 'Epoch', 'Accuracy (%)', 'Model Accuracy')
a1.set_ylim(0, 100)
mark_best(a1)
a1.legend(loc='lower right', frameon=False, prop={'weight': 'bold'})
a2.plot(h['epoch'], h['train_loss'], color=TRAIN_C, lw=2.2, label='Training')
a2.plot(h['epoch'], h['val_loss'], color=VAL_C, lw=2.2, label='Validation')
style(a2, 'Epoch', 'Loss (sum of components)', 'Model Loss')
a2.set_ylim(bottom=0)
mark_best(a2)
a2.legend(loc='best', frameon=False, prop={'weight': 'bold'})
fig.suptitle('BlazeFace (fine-tuned) - training vs validation', fontweight='bold', fontsize=14, y=1.04)
save(fig, 'blazeface_keras_accuracy_loss.png')

# ---------- Extra: training vs validation metrics ----------
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for ax, (key, label) in zip(axes.flat, [('precision', 'Precision'), ('recall', 'Recall'),
                                        ('f1', 'F1-Score'), ('map50', 'mAP@0.5')]):
    ax.plot(h['epoch'], h[f'train_{key}'], color=TRAIN_C, lw=2, label='Training')
    ax.plot(h['epoch'], h[f'val_{key}'], color=VAL_C, lw=2, label='Validation')
    style(ax, 'Epoch' if ax in axes[1] else None, f'{label} (%)', label)
    ax.set_ylim(0, 100)
    mark_best(ax)
    ax.legend(loc='lower right', frameon=False, prop={'weight': 'bold'})
fig.suptitle('BlazeFace (fine-tuned) - training vs validation metrics per epoch', fontweight='bold',
             fontsize=14, y=1.01)
fig.tight_layout()
save(fig, 'blazeface_train_vs_val_metrics.png')

# ---------- Extra: test confusion matrix at the deployed threshold ----------
t = bf_test_df[(bf_test_df.model == 'blazeface') & (bf_test_df.protocol == 'all faces')].iloc[0]
tp, fp, fn = int(t['TP @0.5']), int(t['FP @0.5']), int(t['FN @0.5'])
shade = np.array([[tp, fn], [fp, 0]]) / max(tp + fp + fn, 1)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(shade, cmap='Blues', vmin=0, vmax=1)
for (i, j), text in np.ndenumerate(np.array([[f'TP\n{tp}', f'FN\n{fn}'], [f'FP\n{fp}', 'TN\nn/a*']])):
    ax.text(j, i, text, ha='center', va='center', fontsize=15, fontweight='bold',
            color='white' if shade[i, j] > 0.5 else '#1A1A1A')
ax.set_xticks([0, 1], ['Face', 'Background'])
ax.set_yticks([0, 1], ['Face', 'Background'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('BlazeFace (fine-tuned) - test-set confusion matrix')
fig.text(0.5, -0.02, f'*Detection has no true negatives. MediaPipe default threshold: conf >= {DEPLOY_CONF}, '
         'IoU >= 0.5.', ha='center', fontsize=8.5, color='#555555')
save(fig, 'blazeface_test_confusion_matrix.png')

In [ ]:
# ---------- Extra: original vs fine-tuned, both protocols (TEST) ----------
metrics = [('mAP@0.5', 'mAP50 (%)'), ('mAP@0.5:0.95', 'mAP50-95 (%)'), ('Precision', 'Precision (%)'),
           ('Recall', 'Recall (%)'), ('F1-Score', 'F1 (%)')]
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
for ax, protocol in zip(axes, ('all faces', 'webcam-range faces')):
    for k, name in enumerate(BF_MODELS):
        row = bf_test_df[(bf_test_df.model == name) & (bf_test_df.protocol == protocol)].iloc[0]
        xs = np.arange(len(metrics)) + (k - 0.5) * 0.38
        vals = [row[c] for _, c in metrics]
        ax.bar(xs, vals, 0.35, color=MODEL_COLORS[DISPLAY[name]], edgecolor='black', linewidth=1.1,
               label=DISPLAY[name])
        for x, v in zip(xs, vals):
            ax.text(x, v + 0.8, f'{v:.1f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
    ax.set_xticks(range(len(metrics)), [m for m, _ in metrics], fontweight='bold', fontsize=10)
    style(ax, None, 'Score (%)' if protocol == 'all faces' else None,
          'All faces' if protocol == 'all faces' else f'Webcam-range faces (>= {IN_RANGE_MIN_FACE:.0%} of image)')
    ax.set_ylim(0, 105)
axes[0].legend(loc='upper right', frameon=False, prop={'weight': 'bold'})
save(fig, 'blazeface_original_vs_finetuned.png')

# ---------- Extra: recall by face size (TEST, IoU >= 0.5, conf >= 0.5) ----------
BINS = [0, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20, 0.30, 1.01]
sizes = np.concatenate([face_size(g, lb) for g, lb in zip(test_set['gt'], test_set['lb'])])
which = np.digitize(sizes, BINS) - 1


def recall_by_size(dets, conf=DEPLOY_CONF):
    hit = []
    for (b, s), g in zip(dets, test_set['gt']):
        b = b[s >= conf]
        hit.append(iou_matrix(g, b).max(1) >= 0.5 if len(b) and len(g) else np.zeros(len(g), bool))
    hit = np.concatenate(hit)
    return [hit[which == i].mean() * 100 if (which == i).any() else np.nan for i in range(len(BINS) - 1)]


series = {DISPLAY[n]: recall_by_size(test_dets[n]) for n in BF_MODELS}
fig, ax = plt.subplots(figsize=(13, 5.5))
labels = [f'{BINS[i]:.0%}-{BINS[i + 1]:.0%}'.replace('101%', '100%') for i in range(len(BINS) - 1)]
for k, (label, vals) in enumerate(series.items()):
    xs = np.arange(len(labels)) + (k - 0.5) * 0.38
    ax.bar(xs, vals, 0.35, color=MODEL_COLORS[label], edgecolor='black', linewidth=1.1, label=label)
    for x, v in zip(xs, vals):
        if not np.isnan(v):
            ax.text(x, v + 1, f'{v:.0f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
counts = np.bincount(which, minlength=len(BINS) - 1)[:len(BINS) - 1]
ax.set_xticks(range(len(labels)), [f'{l}\n(n={c})' for l, c in zip(labels, counts)])
style(ax, 'Face size (share of the image\'s longer side)', f'Recall @ conf {DEPLOY_CONF} (%)')
ax.axvline(np.searchsorted(BINS, IN_RANGE_MIN_FACE) - 0.5, color='#5B7A99', ls=':', lw=1.5)
ax.set_ylim(0, 105)
legend_top(ax, ncol=2)
save(fig, 'blazeface_recall_by_face_size.png')

In [ ]:
# ---------- Paper Table 2 / Fig. 48 / Fig. 49: comparison across models (TEST) ----------
def table_png(df, fname, title):
    cols = ['model', 'Type', 'mAP50 (%)', 'mAP50-95 (%)', 'Precision (%)', 'Recall (%)', 'F1 (%)', 'Accuracy (%)',
            'Mean Confidence (%)', 'Inference CPU (ms)', 'FPS CPU', 'Params (M)']
    cols = [c for c in cols if c in df and df[c].notna().any()]
    higher = {'mAP50 (%)', 'mAP50-95 (%)', 'Precision (%)', 'Recall (%)', 'F1 (%)', 'Accuracy (%)',
              'Mean Confidence (%)', 'FPS CPU'}
    best_row = {c: (df[c].idxmax() if c in higher else df[c].idxmin()) for c in cols[2:]}
    headers = [{'model': 'Model', 'mAP50 (%)': 'mAP@0.5\n(%)', 'mAP50-95 (%)': 'mAP@0.5:0.95\n(%)',
                'Mean Confidence (%)': 'Mean\nConfidence (%)', 'Inference CPU (ms)': 'Inference\nCPU (ms)',
                'FPS CPU': 'FPS\nCPU'}.get(c, c.replace(' (', '\n(')) for c in cols]
    cells = [[('' if pd.isna(r[c]) else f'{r[c]:.2f}' if isinstance(r[c], float) else str(r[c])) for c in cols]
             for _, r in df.iterrows()]
    chars = [max(max(len(line) for line in h.split('\n')), *(len(r[j]) for r in cells)) + 2
             for j, h in enumerate(headers)]
    widths = [c / sum(chars) for c in chars]
    fig, ax = plt.subplots(figsize=(0.115 * sum(chars) + 0.4, 0.62 * (len(df) + 1) + 0.6))
    ax.axis('off')
    tbl = ax.table(cellText=cells, colLabels=headers, colWidths=widths, cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 2.3)
    for (i, j), cell in tbl.get_celld().items():
        cell.set_edgecolor('#444444')
        if i == 0 or j == 0:
            cell.set_text_props(fontweight='bold')
        elif j >= 2 and best_row.get(cols[j]) == df.index[i - 1]:
            cell.set_facecolor('#BDD7EE')
            cell.set_text_props(fontweight='bold')
    ax.set_title(title, fontweight='bold', pad=6)
    save(fig, fname)


table_png(compare_all, 'compare_table.png', 'Face-detection algorithms on the same test split (all faces)')
table_png(compare_range, 'compare_table_webcam_range.png',
          f'Same test split, webcam-range faces only (>= {IN_RANGE_MIN_FACE:.0%} of the image side)')


def compare_bars(df, fname):
    metrics = [('mAP@0.5', 'mAP50 (%)'), ('mAP@0.5:0.95', 'mAP50-95 (%)'), ('Precision', 'Precision (%)'),
               ('Recall', 'Recall (%)'), ('F1-Score', 'F1 (%)')]
    width = 0.8 / len(df)
    fig, ax = plt.subplots(figsize=(14, 6))
    for k, (_, r) in enumerate(df.iterrows()):
        xs = np.arange(len(metrics)) + (k - (len(df) - 1) / 2) * width
        vals = [r[c] for _, c in metrics]
        ax.bar(xs, vals, width * 0.92, color=MODEL_COLORS[r['model']], edgecolor='black', linewidth=1.1,
               label=r['model'])
        for x, v in zip(xs, vals):
            ax.text(x, v + 0.8, f'{v:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
    ax.set_xticks(range(len(metrics)), [m for m, _ in metrics], fontweight='bold')
    style(ax, None, 'Score (%)')
    ax.set_ylim(0, 105)
    legend_top(ax, ncol=len(df))
    save(fig, fname)


compare_bars(compare_all, 'compare_test_metrics.png')
compare_bars(compare_range, 'compare_test_metrics_webcam_range.png')

# Accuracy vs speed: every model timed on this runtime's CPU (log scale - BlazeFace is orders of magnitude faster).
speed_df = compare_all.dropna(subset=['FPS CPU'])
fig, ax = plt.subplots(figsize=(13, 6))
xmin, xmax = speed_df['FPS CPU'].min() / 2.5, speed_df['FPS CPU'].max() * 3
ax.set_xscale('log')
ax.axvspan(xmin, REALTIME_FPS, color='#F2D7D7', alpha=0.8, zorder=0)
ax.axvline(REALTIME_FPS, color='#C0504D', ls='--', lw=1.5)
ax.text(REALTIME_FPS * 1.06, 0.97, f'{REALTIME_FPS} FPS - live-webcam real-time threshold',
        transform=ax.get_xaxis_transform(), color='#C0504D', fontweight='bold', va='top')
logx = np.log10(speed_df['FPS CPU'].to_numpy())
for k, (_, r) in enumerate(speed_df.iterrows()):
    x, y, params, c = r['FPS CPU'], r['mAP50 (%)'], r['Params (M)'], MODEL_COLORS[r['model']]
    ax.scatter(x, y, s=max(params, 0.1) * 120 + 40, color=c, alpha=0.35, edgecolor=c, linewidth=2.5, zorder=3)
    ax.scatter(x, y, s=18, color=c, zorder=4)
    # Models at similar speed share a column: the more accurate one is labelled above, the other below.
    near = speed_df['mAP50 (%)'].to_numpy()[np.abs(logx - logx[k]) < 0.35]
    above = y >= near.max() or len(near) == 1
    ax.annotate(f"{r['model']}\n{y:.2f} mAP · {x:.0f} FPS · {params:.2f}M", (x, y),
                xytext=(0, 22 if above else -24), textcoords='offset points', ha='center',
                va='bottom' if above else 'top', color=c, fontweight='bold', fontsize=10, zorder=5)
ax.set_xlim(xmin, xmax)
lo, hi = speed_df['mAP50 (%)'].min(), speed_df['mAP50 (%)'].max()
span = max(hi - lo, 8)
ax.set_ylim(max(0, lo - 0.45 * span), min(105, hi + 0.45 * span))
style(ax, f'Inference speed on {CPU_NAME} (FPS, log scale) -> faster', 'mAP@0.5 on test split (%)')
ax.text(0.99, 0.99, 'bubble area ∝ model size (params)', transform=ax.transAxes, ha='right', va='top', fontsize=9,
        style='italic', color='#666666')
save(fig, 'compare_accuracy_vs_speed.png')
print('Figures saved to', FIG_DIR)
print(sorted(os.listdir(FIG_DIR)))

## 12. What to download and how to use it

Everything is in `MyDrive/knowing-eye-blazeface-70-20-10/`:

* `test_results_blazeface_70_20_10.csv` - **final TEST numbers** for the original and fine-tuned BlazeFace,
  all faces and webcam-range faces
* `comparison_all_faces.csv`, `comparison_webcam_range_faces.csv` - the Table 2 comparison with the YOLO models
* `blazeface/history_checkpoints.csv` - paper Table 1 / Table 3 layout (validation, epochs 20-100 + average);
  `history_checkpoints_extended.csv`, `history_all_epochs.csv` - the full training history
* `figures/` - every chart as a 200-dpi PNG (see the table in section 11)
* `blaze_face_short_range_finetuned.tflite` - copy over `frontend/public/models/blaze_face_short_range.tflite`
* `face_landmarker_finetuned.task` - copy over `backend/ai/models/face_landmarker.task` (the server's face
  pipeline; only the face *detector* inside it changes - landmarks and blendshapes stay Google's)
* `blazeface/best.weights.h5` - the Keras weights the test numbers belong to

**For the paper's methodology:** BlazeFace short-range (MediaPipe; 0.10 M parameters, 128x128 input) was
fine-tuned from Google's pretrained weights on the same WIDER FACE + FDDB 70/20/10 split as the other detectors:
Adam (peak LR 1e-3, warm-up + cosine), batch 32, up to 100 epochs with early stopping (patience 20) on validation
mAP@0.5:0.95; SSD anchor matching (IoU >= 0.4), BCE with 3:1 hard-negative mining, GIoU box loss, and landmark
distillation from the original model; random zoom (0.8-2.5x), flip and colour augmentation. Metrics use the
Ultralytics definitions; test images are processed exactly as MediaPipe's FaceDetector processes them.